### INITIALIZATION: 
IMPORT TOOLS, SET SEED, AND CREATE INDEPENDENT TABLES

In [1]:
# libraries
import pandas as pd
import numpy as np
import uuid
from datetime import datetime, timedelta

# seed
SEED = 42
np.random.seed(SEED)


print("Libraries loaded and seed set.")

Libraries loaded and seed set.


### Independent Entities

In [2]:
# Step 2: Independent Entities (Departments)
departments_data = [
    {"name": "Traffic & Transport", "daily_capacity": 50, "vulnerability_to_storm": 5.0, "base_rate": 20},
    {"name": "Public Works", "daily_capacity": 40, "vulnerability_to_storm": 8.0, "base_rate": 15},
    {"name": "Parks & Recreation", "daily_capacity": 10, "vulnerability_to_storm": 1.5, "base_rate": 2},
    {"name": "Animal Control", "daily_capacity": 15, "vulnerability_to_storm": 1.0, "base_rate": 5},
    {"name": "Sanitation", "daily_capacity": 60, "vulnerability_to_storm": 3.0, "base_rate": 45}
]

df_departments = pd.DataFrame(departments_data)

# Give each department a unique UUID
df_departments['department_id'] = [str(uuid.uuid4()) for _ in range(len(df_departments))]

print(df_departments.head())

                  name  ...                         department_id
0  Traffic & Transport  ...  559cf90b-d623-4d71-8e77-47e5d388d198
1         Public Works  ...  5e8ce920-91e1-4aeb-8fc5-73822dca3c46
2   Parks & Recreation  ...  852f01d2-05df-4b76-916c-33b4b0e2acad
3       Animal Control  ...  19970364-0b93-4df7-ab1c-ddd807e72bbe
4           Sanitation  ...  0885c9d6-b4bd-4e0f-84c3-576c0f3dcee2

[5 rows x 5 columns]


In [3]:
# setup 
NUM_CITIZENS = 10000
BASE_DATE = datetime(2026, 8, 1)

# citizen ids
# generate 10000 ids for each citizen
citizen_ids = [str(uuid.uuid4()) for _ in range(NUM_CITIZENS)]

# dates
# generate random dates
random_days_ago = np.random.randint(1, 365,size=NUM_CITIZENS)
join_dates = [
    (BASE_DATE - timedelta(days=int(days))).strftime("%Y-%m-%d") 
    for days in random_days_ago
]

# clip L_civic
# np.random.normal creates the raw data without clipping 
# np.clip forces any number below 0.1 to become 0.1, and any above 2 to become 2.
raw_scores = np.random.normal(loc=0.5, scale=0.3, size=NUM_CITIZENS)
L_civics = np.clip(raw_scores, a_min=0.1, a_max=2.0)


# creation of data frame
df_citizens = pd.DataFrame({
    "citizen_id": citizen_ids,
    "join_date": join_dates,
    "L_civic": L_civics,
})

print(df_citizens.head())
print("\nL_civic:")
print(df_citizens["L_civic"].describe())

                             citizen_id   join_date   L_civic
0  5bd53cba-3551-49e5-b841-cf0fd6e70062  2026-04-20  0.798459
1  5878fae8-1ada-4f7a-94d7-03b3fba45723  2025-08-17  0.471951
2  3d58b40a-cf5a-4b9c-84c9-9f88a5ed1fc7  2025-11-03  1.152113
3  1185c819-147f-491c-9f92-1f489b5589d7  2026-04-16  0.100000
4  feb5ca03-9f21-4f18-bd0a-53298af6712c  2026-05-21  0.211590

L_civic:
count    10000.000000
mean         0.511578
std          0.275631
min          0.100000
25%          0.296869
50%          0.499840
75%          0.701194
max          1.525584
Name: L_civic, dtype: float64


### Step 3B: Timeline & Latent Weather Shock
Simulate a 30-day calendar starting from  (August 1, 2026) and inject a hidden omitted variable  representing a decaying typhoon shock.

In [4]:
# Step 3B: Timeline & Latent Weather Shock

# 1. 30 consecutive days starting from BASE_DATE using timedelta list comprehension
# create a list of all dates from the starting base date
timeline_dates = [
    (BASE_DATE + timedelta(days=i)).strftime("%Y-%m-%d")
    for i in range(30)
]

# 2. Vectorized initialization of latent storm shock (L_storm)
# intialize a 30 row list with 0.0
L_storm = np.zeros(30)

# 3. Inject decaying typhoon shock at specific indexes (Day 15, Day 16, Day 17)
# change the values to simulate a typoon situation
L_storm[14] = 1.0  # Index 14 (Day 15): Peak typhoon shock
L_storm[15] = 0.6  # Index 15 (Day 16): Receding floodwaters
L_storm[16] = 0.2  # Index 16 (Day 17): Residual shock

# Create timeline DataFrame
df_timeline = pd.DataFrame({
    "date": timeline_dates,
    "L_storm": L_storm
})

# Verify storm pulse injection
print("Timeline created. Storm pulse slice (index 13:18):")
print(df_timeline.iloc[13:18])

Timeline created. Storm pulse slice (index 13:18):
          date  L_storm
13  2026-08-14      0.0
14  2026-08-15      1.0
15  2026-08-16      0.6
16  2026-08-17      0.2
17  2026-08-18      0.0


### Step 3C: Timeline & Department Vulnerability Interaction Grid
Perform a Cartesian cross-join between `df_timeline` (30 days) and `df_departments` (5 departments) to create a 150-row simulation grid (`df_grid`).

In [5]:
# Step 3C: Timeline & Department Vulnerability Interaction Grid

# Perform Cartesian cross-join between timeline and departments
# merge the timeline with departments table, creating 150 rows
df_grid = df_timeline.merge(df_departments, how="cross")

# Verification
print(f"Simulation grid created with shape: {df_grid.shape}")
print("\nSample rows during storm peak (2026-08-15):")
print(df_grid[df_grid["date"] == "2026-08-15"][["date", "name", "L_storm", "vulnerability_to_storm", "base_rate"]])


Simulation grid created with shape: (150, 7)

Sample rows during storm peak (2026-08-15):
          date                 name  L_storm  vulnerability_to_storm  base_rate
70  2026-08-15  Traffic & Transport      1.0                     5.0         20
71  2026-08-15         Public Works      1.0                     8.0         15
72  2026-08-15   Parks & Recreation      1.0                     1.5          2
73  2026-08-15       Animal Control      1.0                     1.0          5
74  2026-08-15           Sanitation      1.0                     3.0         45


### Step 4: Core Ticket Generation Engine
Calculates expected daily ticket volume ($\lambda$) per department using $\lambda = \text{base\_rate} + (\text{base\_rate} \times L_{\text{storm}} \times \text{vulnerability\_to\_storm})$. Samples realized ticket counts using Poisson distribution (`np.random.poisson(lam)`) and assigns tickets to citizens weighted by their civic engagement score $L_{\text{civic}}$ (hyper-reporters).

In [6]:
# Step 4: Core Ticket Generation Engine
# Goal: Simulate how many complaints each department receives daily, and who reports them.

# 1. CALCULATE EXPECTED DAILY TICKETS (lambda / average daily rate)
# Formula: BaseRate + (BaseRate * L_storm * vulnerability_to_storm)
# Why: Sunny days (L_storm=0) keep base rates (e.g. 15). Storm days (L_storm=1.0) surge rates based on vulnerability.
df_grid["lambda"] = df_grid["base_rate"] + (
    df_grid["base_rate"] * df_grid["L_storm"] * df_grid["vulnerability_to_storm"]
)

# 2. GENERATE REALIZED DAILY TICKET COUNTS (Poisson Random Sampling)
# Why Poisson? Real city complaints fluctuate around an average. np.random.poisson(lam) generates realistic random counts.
df_grid["num_tickets"] = np.random.poisson(df_grid["lambda"])

# 3. CALCULATE CITIZEN REPORTING PROBABILITIES (Hyper-Reporter Weights)
# Why: Divide each citizen's L_civic score by the sum of all scores so total probability adds up to 100% (1.0).
citizen_weights = df_citizens["L_civic"].values / df_citizens["L_civic"].sum()

# 4. ASSIGN TICKETS TO CITIZENS (Weighted Random Choice / Raffle Drum)
# Why: np.random.choice picks citizens randomly, giving higher-scoring citizens a greater chance of being picked.
ticket_records = []
for idx, row in df_grid.iterrows():
    count = row["num_tickets"]
    if count > 0:
        assigned_citizens = np.random.choice(
            df_citizens["citizen_id"].values,
            size=count,
            p=citizen_weights
        )
        for c_id in assigned_citizens:
            ticket_records.append({
                "ticket_id": str(uuid.uuid4()),      # Unique 128-bit ID per ticket
                "created_at": row["date"],          # Creation date string (YYYY-MM-DD)
                "department_id": row["department_id"], # Department receiving the ticket
                "citizen_id": c_id                  # Assigned citizen ID
            })

# 5. CONSTRUCT FINAL TICKETS DATAFRAME
df_tickets = pd.DataFrame(ticket_records)

# Verification & Summaries (Pandas Pivot Table Grouping)
print(f"Total synthetic tickets generated: {len(df_tickets)}")
print("\nHead of df_tickets (First 5 rows):")
print(df_tickets.head())
print("\nDaily ticket counts during storm surge (2026-08-13 to 2026-08-18):")
daily_summary = df_grid.groupby("date")["num_tickets"].sum().reset_index()
print(daily_summary.iloc[12:18])


Total synthetic tickets generated: 3199

Head of df_tickets (First 5 rows):
                              ticket_id  ...                            citizen_id
0  19c72b76-61ab-4a3b-be1c-913611fce2a9  ...  09725171-596f-4fa1-98de-06213f514995
1  f7513f49-04a8-4052-bd25-1bcc0bcb834e  ...  312bc6f2-1759-434a-bc90-0a126ccc2678
2  e7679196-216e-491a-ba51-4129ff662137  ...  e70f94ff-817f-4438-acad-0c34eb97751b
3  2b3a3cdc-1ed0-4c47-90c2-83180c1d3a61  ...  6b2a5b24-4a9b-4576-b990-20e8dd9f4213
4  07e6fa49-178d-4883-bf21-7171c3023ccb  ...  81d8031e-d348-4e6a-acf6-837ab4c29d6d

[5 rows x 4 columns]

Daily ticket counts during storm surge (2026-08-13 to 2026-08-18):
          date  num_tickets
12  2026-08-13           87
13  2026-08-14           65
14  2026-08-15          491
15  2026-08-16          275
16  2026-08-17          141
17  2026-08-18           86


### Step 5: SLA Resolution Engine (Log-Normal Distribution)
Generates realistic resolution times in hours (`resolution_time_hours`) using a **Log-Normal distribution** (`np.random.lognormal`). Most tickets are resolved quickly (4–8 hours), but severe storm days trigger a heavy backlog shift that delays resolution times up to 70+ hours.

In [7]:
# Step 5: SLA Resolution Engine
# Goal: Calculate how long each ticket takes to resolve in hours.

# 1. MAP STORM SEVERITY (L_storm) TO TICKETS BY CREATION DATE
# Why: We merge L_storm from df_timeline based on created_at date so storm-day tickets get backlog delays.
df_tickets = df_tickets.merge(df_timeline[["date", "L_storm"]], left_on="created_at", right_on="date", how="left")

# 2. CALCULATE LOG-NORMAL PARAMETERS FOR SKEWED RESOLUTION DELAYS
# Why Log-Normal? Most tickets take 4-8 hours, but storm backlogs cause long right-tail delays (70+ hours).
# Base meanlog = 1.5 (median ~4.5 hours). Storm backlog shifts meanlog upwards by (L_storm * 1.2).
meanlog = 1.5 + (df_tickets["L_storm"] * 1.2)
sigmalog = 0.6

# 3. SAMPLE RESOLUTION TIMES AND ROUND TO 1 DECIMAL PLACE
# np.random.lognormal draws random numbers from the skewed distribution. np.round rounds to 1 decimal digit.
df_tickets["resolution_time_hours"] = np.round(np.random.lognormal(meanlog, sigmalog), 1)

# Verification: Summary statistics comparing normal days vs peak storm days
print("Resolution Time Statistics (Hours) by Storm Level:")
print(df_tickets.groupby("L_storm")["resolution_time_hours"].describe())


Resolution Time Statistics (Hours) by Storm Level:
          count       mean        std  min  25%   50%   75%   max
L_storm                                                          
0.0      2292.0   5.486257   3.566207  0.8  3.1   4.7   7.0  54.0
0.2       141.0   6.621986   4.283258  1.1  3.6   5.7   8.5  31.6
0.6       275.0  10.780000   7.118420  1.2  5.8   8.8  14.2  59.7
1.0       491.0  17.216293  10.531057  1.0  9.8  14.7  22.1  76.1


### Step 6: Omitted Variable Removal & Final Dataset Export
To create a realistic machine learning benchmark, we **drop the hidden causal variables** (`L_storm` weather shock and `L_civic` engagement score) from the public tables, and export `departments.csv`, `citizens.csv`, and `tickets.csv` into the `data/` directory.

In [8]:
# Step 6: Omitted Variable Removal & Export
import os

# 1. DROP LATENT OMITTED VARIABLES (L_storm from tickets, L_civic from citizens)
# Why: .drop(columns=[...]) removes specified columns so hidden causal variables stay omitted in the benchmark.
df_tickets_export = df_tickets.drop(columns=["L_storm", "date"])
df_citizens_export = df_citizens.drop(columns=["L_civic"])
df_departments_export = df_departments[["department_id", "name", "daily_capacity", "vulnerability_to_storm", "base_rate"]]

# 2. CREATE OUTPUT DATA DIRECTORY
# os.makedirs("../data", exist_ok=True) ensures the data directory exists before saving files.
os.makedirs("../data", exist_ok=True)

# 3. EXPORT CLEAN DATAFRAMES TO CSV FILES (Like saving Excel sheets)
# index=False tells pandas not to write row numbers (0, 1, 2...) as an extra column into the CSV.
df_departments_export.to_csv("../data/departments.csv", index=False)
df_citizens_export.to_csv("../data/citizens.csv", index=False)
df_tickets_export.to_csv("../data/tickets.csv", index=False)

# Verification
print("=== SYNTHETIC DATASET GENERATION COMPLETE ===")
print(f"Exported ../data/departments.csv -> Shape: {df_departments_export.shape}")
print(f"Exported ../data/citizens.csv    -> Shape: {df_citizens_export.shape}")
print(f"Exported ../data/tickets.csv     -> Shape: {df_tickets_export.shape}")


=== SYNTHETIC DATASET GENERATION COMPLETE ===
Exported ../data/departments.csv -> Shape: (5, 5)
Exported ../data/citizens.csv    -> Shape: (10000, 2)
Exported ../data/tickets.csv     -> Shape: (3199, 5)
